# Voice clones vs. speaker verification — Kaggle runner

Reproduces every result in `results/` from a clean Kaggle session.

**Session settings (right panel):** Accelerator = **GPU** (note which one you got: it is recorded in
`results/env/versions.json`), Internet = **On** (needs a phone-verified account).

Every long step is **resumable**: if the session dies, re-run the notebook from the top. Finished runs are
skipped (they already have `test_metrics.json`) and training continues from `last.pt`.
At the end, download `/kaggle/working/results.zip` and commit its contents to the repo.

In [ ]:
# ---- edit this one line: your GitHub repository ----
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/speechbrain-voice-clone-vs-verification.git"

# LibriSpeech goes outside the saved working dir so it does not count against the 20 GB output.
# Check free space in the next cells; if /tmp is too small use "/kaggle/working/data" and delete it afterwards.
DATA_ROOT = "/tmp/data"
SEEDS = [0, 1, 2]

In [ ]:
import os
os.chdir("/kaggle/working")
if not os.path.exists("repo"):
    !git clone -q {REPO_URL} repo
else:
    !git -C repo pull -q
%cd /kaggle/working/repo
!git log -1 --oneline

In [ ]:
# Kaggle already has torch/torchaudio with CUDA; only add what is missing.
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working
!python -m spkclone.utils results/env
!pip freeze > results/env/pip_freeze.txt

## 1. Data (dev-clean + test-clean, ~0.7 GB) and tests

In [ ]:
!python -c "from spkclone.data import ensure_librispeech as e; [e(s, '{DATA_ROOT}') for s in ('dev-clean', 'test-clean')]"

In [ ]:
# Unit tests + the encoder-equivalence test (SpkEncoder vs SpeechBrain EncoderClassifier) on real dev-clean speech.
os.environ["SPKCLONE_TEST_WAVS"] = f"{DATA_ROOT}/LibriSpeech/dev-clean"
!python -m pytest -s -q 2>&1 | tee results/env/pytest.txt
!cat results/env/encoder_equivalence.json

## 2. Smoke tests (minutes) — original CLIs still work

Numbers from these cells are **meaningless** (training speakers overlap with evaluation speakers); they only prove
the pipeline runs end to end. Outputs go to `/tmp/smoke*`, not to `results/`.

In [ ]:
!python scripts/speaker_verification_contrastive.py train --config configs/smoke.yaml --root {DATA_ROOT} --out /tmp/smoke_spk

In [ ]:
# pick a 3-15 s dev-clean utterance as a reference voice (consented, public dataset speaker)
import glob, soundfile as sf
ref = next(f for f in sorted(glob.glob(f"{DATA_ROOT}/LibriSpeech/dev-clean/*/*/*.flac")) if 3 <= sf.info(f).duration <= 15)
print(ref, sf.info(ref).duration)
!python scripts/speechbrain_tts1.py single --text "Doctor Smith bought 3 apples." --plot --out /tmp/smoke_tts
!python scripts/speechbrain_tts1.py random --text "It was a sunny day." --plot --out /tmp/smoke_tts
!python scripts/speechbrain_tts1.py clone --ref {ref} --text "This is a short test sentence." --verify --plot --out /tmp/smoke_tts
!python scripts/speaker_verification_contrastive.py verify --ckpt pretrained --a {ref} --b /tmp/smoke_tts/clone_000.wav
!python scripts/speaker_verification_contrastive.py explain --ckpt pretrained --a {ref} --b /tmp/smoke_tts/clone_000.wav --png /tmp/smoke_tts/why.png

## 3. RQ1 — verifier quality

Same fixed trials (eval_seed 0) for every model. Thresholds are calibrated on dev-clean, reported on test-clean.

In [ ]:
# pretrained VoxCeleb ECAPA-TDNN, no fine-tuning
!python -m spkclone.verify evaluate --ckpt pretrained --root {DATA_ROOT} --out results/rq1/pretrained

In [ ]:
# LibriSpeech train-clean-100 (~6.3 GB download, archive deleted after extraction)
!python -c "from spkclone.data import ensure_librispeech as e; e('train-clean-100', '{DATA_ROOT}')"
!df -h /tmp

In [ ]:
# SupCon fine-tuning, 3 seeds (resumable; re-run this cell after a timeout)
for seed in SEEDS:
    !python -m spkclone.train --config configs/rq1_finetune.yaml --root {DATA_ROOT} --seed {seed} --out runs/rq1/finetune_s{seed} --results results/rq1/finetune_s{seed}

In [ ]:
# small ECAPA from scratch, 3 seeds
for seed in SEEDS:
    !python -m spkclone.train --config configs/rq1_scratch.yaml --root {DATA_ROOT} --seed {seed} --out runs/rq1/scratch_s{seed} --results results/rq1/scratch_s{seed}

In [ ]:
# Ablation (fine-tuning, seed 0 only unless time allows): temperature 0.05 / 0.2 (0.1 is the main run),
# and K=2 with P=32 so the batch still has 64 crops (changing K alone would also halve the batch).
for name, extra in [("ablation_T0.05", "--temp 0.05"), ("ablation_T0.2", "--temp 0.2"), ("ablation_K2", "--K 2 --P 32")]:
    !python -m spkclone.train --config configs/rq1_finetune.yaml --root {DATA_ROOT} --seed 0 {extra} --out runs/rq1/{name}_s0 --results results/rq1/{name}_s0

In [ ]:
!python scripts/aggregate_rq1.py results/rq1

## 4. RQ2–RQ4 — cloning fidelity, spoof vulnerability, explanation

TODO: `spkclone/eval_clone.py` is the next step of the project and is not written yet. Cells will be added here.

In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/results.zip results && ls -lh /kaggle/working/results.zip